# NoPOS — Encoder Head Mask Experiments

Trains an encoder-decoder language model on WikiText-103 with different
per-head attention masks in the encoder (`C` = causal, `F` = future, `B` = bidirectional).

**Setup checklist (do this before running):**
1. Runtime → Change runtime type → GPU → **H100** (or A100)
2. Run cells **1 → 2 → 3 → 4** once per session to set up the environment
3. Edit **Cell 5** to choose your experiment (`COND`, `SPEC`, etc.)
4. Run **Cell 6** to train; re-run it to resume after a session restart

Checkpoints and logs are saved to Google Drive and survive session restarts.

In [ ]:
%%bash
# Cell 1 -- Verify GPU and Python 3.8 environment
# Run this after Cell 3 (install) so PyTorch is installed.
set -e

PY38=/usr/bin/python3.10

echo "=== GPU ==="
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

echo ""
echo "=== Python 3.8 + PyTorch ==="
$PY38 - <<'PYEOF'
import torch
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.version.cuda}")
print(f"GPU     : {torch.cuda.get_device_name(0)}")
print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
PYEOF


=== GPU ===
NVIDIA RTX PRO 6000 Blackwell Server Edition, 97887 MiB

=== Python 3.8 + PyTorch ===
PyTorch : 2.12.1+cu130
CUDA    : 13.0
GPU     : NVIDIA RTX PRO 6000 Blackwell Server Edition
VRAM    : 102.0 GB


In [2]:
%%bash
# Cell 2 -- Set up repo & install (uses Python 3.8 from apt-get)
set -e

PY38=/usr/bin/python3.10

DRIVE_ZIP="/content/drive/MyDrive/nopos_experiments/nopos.zip"

if [ ! -d /content/nopos ]; then
    if [ -f "$DRIVE_ZIP" ]; then
        echo "Extracting repo from Drive ..."
        unzip -q "$DRIVE_ZIP" -d /content/
        cd /content/nopos
        git remote set-url origin https://github.com/Anxinal/futureMask.git
    else
        echo "Cloning Anxinal/futureMask ..."
        git clone --branch main https://github.com/Anxinal/futureMask.git /content/nopos
    fi
fi

cd /content/nopos
echo "Syncing to latest origin/main ..."
git fetch origin main
git reset --hard origin/main
echo "Now at: $(git log -1 --oneline)"

echo "Pinning pip to <22.0.4 ..."
$PY38 -m pip install -q "pip<22.0.4"

echo "Installing PyTorch 2.6+ with Blackwell (SM 12.0) support ..."
$PY38 -m pip install -q "torch>=2.7.0"
$PY38 -c "import torch; print('torch:', torch.__version__, '| CUDA:', torch.cuda.is_available())"

echo "Installing numpy (required for Cython build) ..."
$PY38 -m pip install -q numpy

echo "Installing fairseq (editable, bypassing pyproject.toml) ..."
mv pyproject.toml pyproject.toml.bak
$PY38 -m pip install -q -e . --no-build-isolation
mv pyproject.toml.bak pyproject.toml

$PY38 -c "import fairseq; print('fairseq OK:', fairseq.__version__)"
echo "--- Setup complete ---"

Cloning Anxinal/futureMask ...
Syncing to latest origin/main ...
HEAD is now at 5049644 Merge remote-tracking branch 'origin/main'
Now at: 5049644 Merge remote-tracking branch 'origin/main'
Pinning pip to <22.0.4 ...
Installing PyTorch 2.6+ with Blackwell (SM 12.0) support ...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 526.6/526.6 MB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 KB 9.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 104.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.7/6.7 MB 101.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.0/206.0 MB 12.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.6/197.6 MB 4.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 KB 9.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.9/203.9 KB 43.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.2/366.2 MB 6.9 MB/s eta 0:00:

Cloning into '/content/nopos'...
From https://github.com/Anxinal/futureMask
 * branch            main       -> FETCH_HEAD
/usr/local/lib/python3.10/dist-packages/torch/_subclasses/functional_tensor.py:368: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /__w/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))
2026-07-12 04:47:30 | INFO | fairseq.tasks.text_to_speech | Please install tensorboardX: pip install tensorboardX


In [1]:
#cell 3 -- Install Python 3.10
# 1. Install Python 3.10 and dev headers
!sudo apt-get update -y -qq
!sudo apt-get install -y -qq python3.10 python3.10-dev python3.10-distutils

# 2. Bootstrap pip for Python 3.10
!curl -sS https://bootstrap.pypa.io/get-pip.py | sudo /usr/bin/python3.10

!/usr/bin/python3.10 --version
print("Python 3.10 ready at /usr/bin/python3.10")


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 7.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
(Reading database ... 122403 files and directories currently installed.)
Preparing to unpack .../0-libpython3.10-dev_3.10.12-1~22.04.16_amd64.deb ...
Unpacking libpython3.10-dev:amd64 (3.10.12-1~22.04.16) over (3.10.12-1~22.04.15) ...
Preparing to unpack .../1-libpython3.10_3.10.12-1~22.04.16_amd64.deb ...
Unpacking libpython3.10:amd64 (3.10.12-1~22.04.16) ove

In [24]:
# Cell 4 — Mount Google Drive
# Checkpoints and logs are stored here so they survive session restarts.
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/nopos_experiments'
os.makedirs(f'{DRIVE_DIR}/checkpoints', exist_ok=True)
os.environ['DRIVE_DIR'] = DRIVE_DIR   # make available to %%bash cells
print(f"Drive mounted. All outputs → {DRIVE_DIR}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted. All outputs → /content/drive/MyDrive/nopos_experiments


In [23]:
%%bash
# Cell 5 — Download & preprocess WikiText-103
set -e

PY38=/usr/bin/python3.10
DATABIN="/content/nopos/data-bin/wikitext-103"
DRIVE_DATABIN="${DRIVE_DIR}/data-bin/wikitext-103"

if [ -d "$DATABIN" ] && [ -n "$(ls -A $DATABIN 2>/dev/null)" ]; then
    echo "Preprocessed data already in /content — nothing to do."

elif [ -d "$DRIVE_DATABIN" ]; then
    echo "Restoring preprocessed data from Drive ..."
    mkdir -p "$(dirname $DATABIN)"
    cp -r "$DRIVE_DATABIN" "$DATABIN"
    echo "Restored."

else
    echo "Downloading WikiText-103 via HuggingFace datasets ..."
    $PY38 -m pip install -q datasets
    mkdir -p /content/wt103-raw/wikitext-103
    $PY38 /content/nopos/scripts/download_wikitext.py

    echo "Preprocessing (dictionary build + binarise) ..."
    cd /content/nopos
    $PY38 -m fairseq_cli.preprocess --only-source --trainpref /content/wt103-raw/wikitext-103/wiki.train.tokens --validpref /content/wt103-raw/wikitext-103/wiki.valid.tokens --testpref /content/wt103-raw/wikitext-103/wiki.test.tokens --destdir "$DATABIN" --workers 4

    echo "Caching to Drive for future sessions ..."
    mkdir -p "$(dirname $DRIVE_DATABIN)"
    cp -r "$DATABIN" "$(dirname $DRIVE_DATABIN)/"
    echo "Done."
fi

echo ""
echo "Contents of data-bin:"
ls /content/nopos/data-bin/wikitext-103/


Preprocessed data already in /content — nothing to do.

Contents of data-bin:
dict.txt
preprocess.log
test.bin
test.idx
train.bin
train.idx
valid.bin
valid.idx


In [29]:
# ╔══════════════════════════════════════════════════════════════╗
# ║           Cell 6 — CONFIGURE YOUR EXPERIMENT               ║
# ║   Edit the values in this cell, then run Cell 6 to train   ║
# ╚══════════════════════════════════════════════════════════════╝

# ── Experiment identity ─────────────────────────────────────────
COND = "BBBBBBBBdim1024-1"           # Name used for log/checkpoint files
SPEC = "B,B,B,B,B,B,B,B"   # Encoder head mask per head (8 heads total)
                             #   C = causal (attend to past only)
                             #   F = future (attend to future only)
                             #   B = bidirectional (unrestricted)

# ── Batch & compute ─────────────────────────────────────────────
# Local baseline used max_tokens=2048 on RTX 4060.
# H100 can handle much larger batches — increase MAX_TOKENS for
# faster throughput. Effective batch = MAX_TOKENS * UPDATE_FREQ.
# To keep the same update dynamics as local, keep MAX_TOKENS=2048.
MAX_TOKENS    = 4096    # tokens per GPU step (increase on H100, e.g. 8192)
UPDATE_FREQ   = 1       # gradient accumulation steps
MAX_UPDATES   = 80000  # total gradient updates
VALIDATE_EVERY = 2000   # run validation every N updates
SAVE_EVERY     = 10000   # checkpoint every N updates

# ── Derived paths (no need to edit) ─────────────────────────────
import os
DRIVE_DIR = os.environ.get('DRIVE_DIR', '/content/drive/MyDrive/nopos_experiments')
SAVE_DIR  = f'{DRIVE_DIR}/checkpoints/{COND}'
LOG_FILE  = f'{DRIVE_DIR}/{COND}.log'
os.makedirs(SAVE_DIR, exist_ok=True)

print(f"Condition    : {COND}")
print(f"Head mask    : {SPEC}")
print(f"Max tokens   : {MAX_TOKENS}  (update_freq={UPDATE_FREQ})")
print(f"Max updates  : {MAX_UPDATES}")
print(f"Validate every {VALIDATE_EVERY} | Save every {SAVE_EVERY}")
print(f"Save dir     : {SAVE_DIR}")
print(f"Log file     : {LOG_FILE}")

ckpt = os.path.join(SAVE_DIR, 'checkpoint_last.pt')
if os.path.exists(ckpt):
    print(f"\nFound existing checkpoint — Cell 6 will RESUME from it.")
else:
    print("\nNo existing checkpoint — Cell 6 will start fresh.")

Condition    : BBBBBBBBdim1024-1
Head mask    : B,B,B,B,B,B,B,B
Max tokens   : 4096  (update_freq=1)
Max updates  : 80000
Validate every 2000 | Save every 10000
Save dir     : /content/drive/MyDrive/nopos_experiments/checkpoints/BBBBBBBBdim1024-1
Log file     : /content/drive/MyDrive/nopos_experiments/BBBBBBBBdim1024-1.log

No existing checkpoint — Cell 6 will start fresh.


In [32]:
# Cell 6 — Run training
# Re-run this cell after a session restart to resume from the Drive checkpoint.
import subprocess, json, re, os
from datetime import datetime

DATABIN = '/content/nopos/data-bin/wikitext-103'

cmd = [
    '/usr/bin/python3.10', '-m', 'fairseq_cli.train', DATABIN,
    '--task',                         'encoder_decoder_language_modeling',
    '--sample-break-mode',            'none',
    '--tokens-per-sample',            '512',
    '--encoder-prefix-fraction',      '0.5',
    '--arch',                         'transformer',
    '--encoder-layers',               '8',
    '--decoder-layers',               '8',
    '--encoder-attention-heads',      '8',
    '--decoder-attention-heads',      '8',
    '--encoder-embed-dim',            '1024',
    '--decoder-embed-dim',            '1024',
    '--encoder-ffn-embed-dim',        '4096',
    '--decoder-ffn-embed-dim',        '4096',
    '--share-all-embeddings',
    '--no-token-positional-embeddings',
    '--encoder-head-mask-spec',       SPEC,
    '--dropout',                      '0.1',
    '--attention-dropout',            '0.1',
    '--optimizer',                    'adam',
    '--adam-betas',                   '(0.9, 0.98)',
    '--weight-decay',                 '0.01',
    '--clip-norm',                    '1.0',
    '--lr',                           '1e-4',
    '--lr-scheduler',                 'inverse_sqrt',
    '--warmup-updates',               '12000',
    '--criterion',                    'cross_entropy',
    '--max-tokens',                   str(MAX_TOKENS),
    '--update-freq',                  str(UPDATE_FREQ),
    '--max-update',                   str(MAX_UPDATES),
    '--skip-invalid-size-inputs-valid-test',
    '--required-batch-size-multiple', '1',
    '--validate-interval-updates',    str(VALIDATE_EVERY),
    '--save-interval-updates',        str(SAVE_EVERY),
    '--keep-interval-updates',        '1',
    '--keep-best-checkpoints',        '1',
    '--no-epoch-checkpoints',
    '--log-interval',                 '100',
    '--log-format',                   'json',
    '--num-workers',                  '4',
    '--seed',                         '1',
    '--save-dir',                     SAVE_DIR,
]

print('=' * 65)
print(f'  Condition   : {COND}')
print(f'  Head mask   : {SPEC}')
print(f'  Batch       : max_tokens={MAX_TOKENS} x update_freq={UPDATE_FREQ}')
print(f'  Max updates : {MAX_UPDATES}')
print(f'  Save dir    : {SAVE_DIR}')
print(f'  Log file    : {LOG_FILE}')
ckpt = os.path.join(SAVE_DIR, 'checkpoint_best.pt')
print(f'  Resuming    : {os.path.exists(ckpt)}')
print('=' * 65 + '\n')

best_ppl, last_train_upd = float('inf'), -1

def on_line(line, log_fh):
    global best_ppl, last_train_upd
    log_fh.write(line)
    log_fh.flush()
    m = re.search(r'\| (train|valid) \| (\{.+\})\s*$', line)
    if not m:
        if line.strip():
            print(line, end='')  # show errors/warnings
        return
    kind = m.group(1)
    try:
        row = json.loads(m.group(2))
    except json.JSONDecodeError:
        return
    ts = datetime.now().strftime('%H:%M:%S')
    if kind == 'train':
        upd = int(row.get('train_num_updates', 0))
        if upd - last_train_upd >= 1000:
            wps = float(row.get('train_wps', 0))
            print(f'[{ts}] train | upd={upd:>7} | loss={row.get("train_loss"):>6} | '
                  f'ppl={row.get("train_ppl"):>8} | wps={wps:>8.0f}')
            last_train_upd = upd
    elif kind == 'valid':
        upd = int(row.get('valid_num_updates', 0))
        ppl = float(row.get('valid_ppl', 'inf'))
        flag = '  ★ BEST' if ppl < best_ppl else ''
        if upd > 5000 and ppl > 1000:
            print('Error detected: Validation PPL > 1000 after 5000 updates. Check for divergence.')
            os.system.exit(1)  # Terminate the training process due to divergence
            return
        best_ppl = min(best_ppl, ppl)
        print(f'[{ts}] VALID | upd={upd:>7} | loss={row.get("valid_loss"):>6} | ppl={ppl:>8.2f}{flag}')

with open(LOG_FILE, 'a') as log_fh:
    env = {**os.environ, 'CUDA_LAUNCH_BLOCKING': '1'}
    proc = subprocess.Popen(
        cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1, cwd='/content/nopos', env=env
    )
    try:
        for line in proc.stdout:
            on_line(line, log_fh)
    except KeyboardInterrupt:
        proc.terminate()
        print(f'\nInterrupted. Best PPL so far: {best_ppl:.2f}')
        print(f'Checkpoint saved to {SAVE_DIR}')
    ret = proc.wait()
if ret != 0:
    print("[ERROR] Training process exited with code", ret)


if best_ppl < float('inf'):
    print(f'\nDone. Best valid PPL = {best_ppl:.2f}')
print(f'Log → {LOG_FILE}')

  Condition   : BBBBBBBBdim1024-1
  Head mask   : B,B,B,B,B,B,B,B
  Batch       : max_tokens=4096 x update_freq=1
  Max updates : 80000
  Save dir    : /content/drive/MyDrive/nopos_experiments/checkpoints/BBBBBBBBdim1024-1
  Log file    : /content/drive/MyDrive/nopos_experiments/BBBBBBBBdim1024-1.log
  Resuming    : True

2026-07-12 11:27:09 | INFO | fairseq.tasks.text_to_speech | Please install tensorboardX: pip install tensorboardX
2026-07-12 11:27:10 | INFO | fairseq_cli.train | {'_name': None, 'common': {'_name': None, 'no_progress_bar': False, 'log_interval': 100, 'log_format': 'json', 'log_file': None, 'tensorboard_logdir': None, 'wandb_project': None, 'azureml_logging': False, 'seed': 1, 'cpu': False, 'tpu': False, 'bf16': False, 'memory_efficient_bf16': False, 'fp16': False, 'memory_efficient_fp16': False, 'fp16_no_flatten_grads': False, 'fp16_init_scale': 128, 'fp16_scale_window': None, 'fp16_scale_tolerance': 0.0, 'on_cpu_convert_precision': False, 'min_loss_scale': 0.0001, '

: 

In [31]:
# Cell 7 — View validation results from log
import json, re, os

log_to_show = LOG_FILE  # or set to any path, e.g. f'{DRIVE_DIR}/CCCCBBBB.log'

valid_rows, best_ppl = [], float('inf')
with open(log_to_show) as f:
    for line in f:
        m = re.search(r'\| valid \| (\{.+\})\s*$', line)
        if m:
            try:
                valid_rows.append(json.loads(m.group(1)))
            except json.JSONDecodeError:
                pass

if not valid_rows:
    print('No validation entries found in log yet.')
else:
    cond_name = os.path.basename(log_to_show).replace('.log', '')
    print(f'Condition : {cond_name}')
    print(f'{"Updates":>10}  {"Valid Loss":>12}  {"Valid PPL":>12}')
    print('-' * 42)
    for r in valid_rows:
        upd  = int(r.get('valid_num_updates', 0))
        loss = float(r.get('valid_loss', 0))
        ppl  = float(r.get('valid_ppl', 0))
        marker = '  <- BEST' if ppl < best_ppl else ''
        best_ppl = min(best_ppl, ppl)
        print(f'{upd:>10}  {loss:>12.4f}  {ppl:>12.2f}{marker}')
    print(f'\nBest valid PPL : {best_ppl:.2f}')

Condition : BBBBBBBBdim1024-1
   Updates    Valid Loss     Valid PPL
------------------------------------------
      2000        9.6090        780.84  <- BEST
      4000        8.9060        479.54  <- BEST
      6000        8.4990        361.89  <- BEST
      8000        8.1740        288.74  <- BEST
     10000        7.9140        241.24  <- BEST
     12000        7.6600        202.20  <- BEST
     14000        7.5210        183.67  <- BEST
     16000        7.3580        164.02  <- BEST
     18000        7.2350        150.66  <- BEST
     20000        7.1310        140.21  <- BEST
     22000        7.0860        135.84  <- BEST
     24000        7.0020        128.13  <- BEST
     25487        6.9450        123.22  <- BEST
     22000        7.0860        135.84
     24000        7.0020        128.13
     25487        6.9450        123.22
      2000        9.6090        780.84
      4000        8.9060        479.54
      6000        8.4990        361.89
      8000        8.1740      

In [ ]:
# Cell 8 — (Optional) Run multiple conditions sequentially
# Useful for overnight runs. Each condition resumes from Drive if interrupted.
import subprocess, re, json, os
from datetime import datetime

CONDITIONS = [
    # (COND,          SPEC)
    ('BBBBBBBB',  'B,B,B,B,B,B,B,B'),
    ('CFBBBBBB',  'C,F,B,B,B,B,B,B'),
    ('CCFFBBBB',  'C,C,F,F,B,B,B,B'),
    ('CCCCBBBB',  'C,C,C,C,B,B,B,B'),
    ('CCCCFFFF',  'C,C,C,C,F,F,F,F'),
]

DATABIN        = '/content/nopos/data-bin/wikitext-103'
DRIVE_DIR      = os.environ.get('DRIVE_DIR', '/content/drive/MyDrive/nopos_experiments')
MAX_TOKENS_ALL = 4096   # change to 8192 to use H100's full capacity
UPDATE_FREQ_ALL = 1
MAX_UPDATES_ALL = 60000

def build_cmd(cond, spec, save_dir):
    return [
        '/usr/bin/python3.10', '-m', 'fairseq_cli.train', DATABIN,
        '--task',                         'encoder_decoder_language_modeling',
        '--sample-break-mode',            'none',
        '--tokens-per-sample',            '512',
        '--encoder-prefix-fraction',      '0.5',
        '--arch',                         'transformer',
        '--encoder-layers',               '8',
        '--decoder-layers',               '8',
        '--encoder-attention-heads',      '8',
        '--decoder-attention-heads',      '8',
        '--encoder-embed-dim',            '512',
        '--decoder-embed-dim',            '512',
        '--encoder-ffn-embed-dim',        '2048',
        '--decoder-ffn-embed-dim',        '2048',
        '--share-all-embeddings',
        '--no-token-positional-embeddings',
        '--encoder-head-mask-spec',       spec,
        '--dropout',                      '0.1',
        '--attention-dropout',            '0.1',
        '--optimizer',                    'adam',
        '--adam-betas',                   '(0.9, 0.98)',
        '--weight-decay',                 '0.01',
        '--clip-norm',                    '1.0',
        '--lr',                           '5e-4',
        '--lr-scheduler',                 'inverse_sqrt',
        '--warmup-updates',               '5000',
        '--criterion',                    'cross_entropy',
        '--max-tokens',                   str(MAX_TOKENS_ALL),
        '--update-freq',                  str(UPDATE_FREQ_ALL),
        '--max-update',                   str(MAX_UPDATES_ALL),
        '--skip-invalid-size-inputs-valid-test',
        '--required-batch-size-multiple', '1',
        '--validate-interval-updates',    '2000',
        '--no-save',
        '--log-interval',                 '100',
        '--log-format',                   'json',
        '--num-workers',                  '4',
        '--seed',                         '1',
        '--save-dir',                     save_dir,
    ]

summary = []

for cond, spec in CONDITIONS:
    save_dir = f'{DRIVE_DIR}/checkpoints/{cond}'
    log_file = f'{DRIVE_DIR}/{cond}.log'
    os.makedirs(save_dir, exist_ok=True)

    ckpt = os.path.join(save_dir, 'checkpoint_last.pt')
    ts0 = datetime.now().strftime('%H:%M:%S')
    print(f'\n[{ts0}] Starting {cond} (spec={spec}, resume={os.path.exists(ckpt)})')

    best_ppl, last_upd = float('inf'), -1

    with open(log_file, 'a') as log_fh:
        proc = subprocess.Popen(
            build_cmd(cond, spec, save_dir),
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1, cwd='/content/nopos'
        )
        for line in proc.stdout:
            log_fh.write(line)
            log_fh.flush()
            m = re.search(r'\| (train|valid) \| (\{.+\})\s*$', line)
            if not m:
                continue
            kind = m.group(1)
            try:
                row = json.loads(m.group(2))
            except json.JSONDecodeError:
                continue
            ts = datetime.now().strftime('%H:%M:%S')
            if kind == 'train':
                upd = int(row.get('train_num_updates', 0))
                if upd - last_upd >= 5000:
                    print(f'  [{ts}] train upd={upd} ppl={row.get("train_ppl")}')
                    last_upd = upd
            elif kind == 'valid':
                upd = int(row.get('valid_num_updates', 0))
                ppl = float(row.get('valid_ppl', 'inf'))
                flag = ' ★' if ppl < best_ppl else ''
                best_ppl = min(best_ppl, ppl)
                print(f'  [{ts}] VALID upd={upd} ppl={ppl:.2f}{flag}')
        proc.wait()

    summary.append((cond, spec, best_ppl))
    print(f'  Finished {cond} — best PPL = {best_ppl:.2f}')

print('\n' + '=' * 55)
print(f'{"Condition":<14}  {"Spec":<22}  {"Best Valid PPL":>14}')
print('-' * 55)
for cond, spec, ppl in sorted(summary, key=lambda x: x[2]):
    print(f'{cond:<14}  {spec:<22}  {ppl:>14.2f}')
print('=' * 55)


[06:02:51] Starting BBBBBBBB (spec=B,B,B,B,B,B,B,B, resume=False)
  [06:07:34] VALID upd=2000 ppl=1742.19 ★
  [06:12:10] VALID upd=4000 ppl=1652.64 ★
  [06:16:47] VALID upd=6000 ppl=1596.15 ★
  [06:21:23] VALID upd=8000 ppl=1577.03 ★
  [06:25:59] VALID upd=10000 ppl=1566.80 ★
  [06:30:36] VALID upd=12000 ppl=1554.34 ★
  [06:35:12] VALID upd=14000 ppl=1565.64
  [06:39:48] VALID upd=16000 ppl=1547.71 ★
  [06:44:25] VALID upd=18000 ppl=1552.41
  [06:49:01] VALID upd=20000 ppl=1547.57 ★
  [06:53:38] VALID upd=22000 ppl=1549.95
  [06:58:14] VALID upd=24000 ppl=1551.20
  [07:01:40] VALID upd=25487 ppl=1552.24
  [07:01:40] train upd=25487 ppl=1743.74
  [07:02:53] VALID upd=26000 ppl=1546.99 ★
  [07:07:29] VALID upd=28000 ppl=1542.27 ★
  [07:12:05] VALID upd=30000 ppl=1544.87
  [07:16:42] VALID upd=32000 ppl=1537.33 ★
  [07:21:18] VALID upd=34000 ppl=1537.43
  [07:25:55] VALID upd=36000 ppl=1537.07 ★
  [07:30:31] VALID upd=38000 ppl=1539.60
  [07:35:07] VALID upd=40000 ppl=1545.67
  [07:39:44

KeyboardInterrupt: 